In [2]:
import numpy as np
import math
from typing import Callable

In [ ]:
def get_newton_cotes_weights(n: int, a: float, b: float) -> np.ndarray:
    h = (b - a) / n

    # 1. q(s) = (s-0)...(s-n)

    q_coeffs = np.array([1.0])
    for j in range(n + 1):
        q_coeffs = np.convolve(q_coeffs, [1, -j])

    weights = np.zeros(n + 1)

    # 2. computing Ak weights

    for k in range((n//2)  + 1):
        # qs / (s-k)
        poly_k, _ = np.polydiv(q_coeffs, [1, -k])

        powers = np.arange(len(poly_k))[::-1]

        integral = np.sum(poly_k * (n ** (powers + 1)) / (powers + 1))

        factor = h * ((-1)**(n - k)) / (math.factorial(k) * math.factorial(n - k))
        val = factor * integral
        weights[k] = val
        weights[n - k] = val

    return weights # Ak weights

def quadrature(f: Callable[[float], float], n: int, a: float, b: float) -> float:
    x_nodes = np.linspace(a, b, n + 1)
    weights = get_newton_cotes_weights(n, a, b)
    return np.dot(weights, f(x_nodes))

funcs = [
    (lambda x: np.sin(5*x), -1, 9, "sin(5x)"),
    (lambda x: x**-1, 1, 5, "1/x"),
    (lambda x: 1/(1+x**2), -7, 7, "1/(1+x^2)")
]

# ground truth values
exact_vals = [
    (-1/5) * (np.cos(45) - np.cos(-5)), # integral sin(5x)
    np.log(5),                          # integral 1/x
    2 * np.arctan(7)                    # integral 1/(1+x^2)
]

print(f"{'n':<4} | {'sin(5x)':<15} | {'1/x':<15} | {'1/(1+x^2)':<15}")
print("-" * 55)

for n in range(2, 36):
    results = []
    for (f, a, b, _), exact in zip(funcs, exact_vals):
        approx = quadrature(f, n, a, b)
        error = abs(approx - exact)
        results.append(error)

    print(f"{n:<4} | {results[0]:.2e}        | {results[1]:.2e}        | {results[2]:.2e}")

n    | sin(5x)         | 1/x             | 1/(1+x^2)      
-------------------------------------------------------
2    | 9.15e+00        | 7.95e-02        | 6.57e+00
3    | 8.47e-01        | 4.25e-02        | 1.16e+00
4    | 9.15e+00        | 8.34e-03        | 1.96e-01
5    | 1.47e+00        | 5.29e-03        | 7.87e-01
6    | 9.19e-01        | 1.27e-03        | 2.16e+00
7    | 3.58e-01        | 8.60e-04        | 2.55e-02
8    | 9.15e+00        | 2.27e-04        | 2.39e+00
9    | 4.18e-01        | 1.60e-04        | 5.95e-01
10   | 2.73e-01        | 4.49e-05        | 4.51e+00
11   | 8.57e-01        | 3.24e-05        | 8.45e-01
12   | 8.11e+00        | 9.45e-06        | 8.25e+00
13   | 1.87e+01        | 6.99e-06        | 1.73e+00
14   | 1.23e+02        | 1.08e-06        | 1.63e+01
15   | 3.52e+01        | 1.01e-05        | 3.43e+00
16   | 2.36e+11        | 3.14e+07        | 1.58e+10
17   | 1.75e+12        | 1.52e+11        | 1.86e+12
18   | 5.11e+13        | 2.05e+10        | 2.98e+12
1